# Data Explorer: PubSub OHLCV

Analyze PubSub crawl data with OHLCV candlestick charts.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import pandas as pd
import datetime

# import mplfinance as mpf  # Uncomment if installed

## Configuration

In [43]:
# Config
PUBSUB_DIR = "./data/pubsub"

# Stock to analyze
STOCK = "41I1G3000"

# Time interval for OHLCV (e.g., '1min', '5min', '15min', '1h')
TIME_INTERVAL = '1min'

## Load Data

In [44]:
# Find latest date with data
def get_latest_date(data_dir, stock):
    stock_dir = os.path.join(data_dir, stock)
    if not os.path.exists(stock_dir):
        return None
    files = glob.glob(os.path.join(stock_dir, "*.fea"))
    if not files:
        return None
    dates = [os.path.splitext(os.path.basename(f))[0] for f in files]
    print(f"Found dates: {dates}")
    return max(dates)

latest_date = get_latest_date(PUBSUB_DIR, STOCK)
print(f"Latest date: {latest_date}")

Found dates: ['2026-03-16', '2026-03-13']
Latest date: 2026-03-16


In [45]:
# Load PubSub data
if latest_date:
    pubsub_path = os.path.join(PUBSUB_DIR, STOCK, f"{latest_date}.fea")
    df = pd.read_feather(pubsub_path)
    print(f"Loaded {len(df)} rows from {pubsub_path}")
else:
    df = None
    print("No data found")

Loaded 548380 rows from ./data/pubsub/41I1G3000/2026-03-16.fea


In [ ]:
def clean_data(df):
    if df is None or df.empty:
        return None
    
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], format="ISO8601") + pd.Timedelta(hours=7)
    tick_times = df['timestamp'].dt.time
    
    # Define valid market windows
    morning_valid = (tick_times >= datetime.time(9, 0, 0)) & (tick_times <= datetime.time(11, 30, 0))
    afternoon_valid = (tick_times >= datetime.time(13, 0, 0)) & (tick_times <= datetime.time(14, 30, 0))
    
    # Keep only rows that fall in the morning OR afternoon session
    df = df[morning_valid | afternoon_valid]
    # ---------------------------------------------------------
    
    df = df.set_index('timestamp')
    df = df.sort_values(by='total_volume')

    # calculate the match_volume for each row by checking if the total_volume is more than the previous row, default True for the first row
    df['is_match'] = df['total_volume'] > df['total_volume'].shift(1, fill_value=0)
    return df

clean_df = clean_data(df)
# clean_df.head()
# count number of rows where is_match is True
match_count = clean_df['is_match'].sum()
print(f"Number of matches: {match_count}")
# count the sum of match_volume where is_match is True
total_match_volume = clean_df.loc[clean_df['is_match'], 'match_volume'].sum()
print(f"Total match volume: {total_match_volume}")

Number of matches: 93278
Total match volume: 249178


## OHLCV Converter

In [ ]:

def tick_to_ohlcv(df, interval='1min', price_col='match_price', volume_col='match_volume'):
    """
    Convert tick data to OHLCV candlestick data, filtering valid market hours.
    """
    if df is None or df.empty:
        return None
    
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], format="ISO8601") + pd.Timedelta(hours=7)
    tick_times = df['timestamp'].dt.time
    
    # Define valid market windows
    morning_valid = (tick_times >= datetime.time(9, 0, 0)) & (tick_times <= datetime.time(11, 30, 0))
    afternoon_valid = (tick_times >= datetime.time(13, 0, 0)) & (tick_times <= datetime.time(14, 30, 0))
    
    # Keep only rows that fall in the morning OR afternoon session
    df = df[morning_valid | afternoon_valid]
    # ---------------------------------------------------------
    
    df = df.set_index('timestamp')
    
    # pandas .ohlc() automatically creates 'open', 'high', 'low', 'close'
    ohlcv = df[price_col].resample(interval).ohlc()
    ohlcv['volume'] = df[volume_col].resample(interval).sum()
    
    # Drop periods with no trading activity
    ohlcv = ohlcv.dropna(subset=['open']).reset_index()
    
    return ohlcv

## Visualize OHLCV

In [48]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Convert to OHLCV
if df is not None:
    ohlcv = tick_to_ohlcv(df, interval=TIME_INTERVAL)
    print(f"OHLCV - {TIME_INTERVAL}: {len(ohlcv)} candles")
    
    # Determine Volume bar colors (Green for bullish, Red for bearish)
    volume_colors = ['#26a69a' if row['close'] >= row['open'] else '#ef5350' for _, row in ohlcv.iterrows()]

    # 2. Create interactive subplots (Row 1 = Candlestick, Row 2 = Volume)
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.03, 
        subplot_titles=(f'{STOCK} Price', 'Volume'), 
        row_width=[0.2, 0.7]  # 70% height for price, 20% for volume
    )

    # 3. Add Candlestick Trace
    fig.add_trace(
        go.Candlestick(
            x=ohlcv['timestamp'],
            open=ohlcv['open'],
            high=ohlcv['high'],
            low=ohlcv['low'],
            close=ohlcv['close'],
            name='Price',
            increasing_line_color='#26a69a', # Custom green
            decreasing_line_color='#ef5350'  # Custom red
        ),
        row=1, col=1
    )

    # 4. Add Volume Trace
    fig.add_trace(
        go.Bar(
            x=ohlcv['timestamp'],
            y=ohlcv['volume'], 
            name='Volume',
            marker_color=volume_colors
        ),
        row=2, col=1
    )

    # 5. Format layout
    fig.update_layout(
        title=f"{STOCK} - {TIME_INTERVAL} Interactive OHLCV Chart",
        xaxis_rangeslider_visible=False, # Hides the redundant rangeslider at the bottom
        height=700,
        template='plotly_white',
        showlegend=False
    )
    
    # Tweak y-axis to not start at 0 for price
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)

    # Add this right before fig.show() to hide the lunch gap!
    fig.update_xaxes(
        rangebreaks=[
            dict(bounds=[11.5, 13], pattern="hour")
        ]
    )

    fig.show()
else:
    print("No OHLCV data to plot")

OHLCV - 1min: 239 candles
